In [ ]:
# location 컬럼을 기준으로 그룹화하여 전체 고객 수와 이탈 고객 수 계산
state_churn_analysis = music_df.groupby('location')['churned'].agg(['count', 'sum']).reset_index()

# 컬럼명 변경
state_churn_analysis.columns = ['state', 'total_customers', 'churned_customers']

# 이탈률 계산
# 이탈률 = (이탈 고객 수 / 전체 고객 수)
state_churn_analysis['churn_rate'] = state_churn_analysis['churned_customers'] / state_churn_analysis['total_customers']

# 이탈률을 퍼센트로 변환하고 소수점 둘째 자리까지 반올림
state_churn_analysis['churn_rate_pct'] = round(state_churn_analysis['churn_rate'] * 100, 2)

# 이탈률이 높은 순서대로 정렬
state_churn_analysis = state_churn_analysis.sort_values(by='churn_rate_pct', ascending=False)

# 결과 출력
print("State별 이탈률 분석 결과:")
display(state_churn_analysis.head(10))

In [ ]:
!pip install folium

In [ ]:
# 공백 제거 및 첫 글자만 대문자로 통일
state_churn_analysis['state'] = state_churn_analysis['state'].str.strip().str.title()

In [ ]:
# 미국 50개 주 전체 리스트
us_states_full = [
    'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut',
    'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa',
    'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan',
    'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
    'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio',
    'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota',
    'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia',
    'Wisconsin', 'Wyoming'
]

# 데이터에 없는 주 출력
missing_states = [s for s in us_states_full if s not in state_churn_analysis['state'].values]
print("데이터에 없어 색이 안 칠해지는 주:", missing_states)

In [ ]:
import folium
import json
import requests

# 1. 미국 주 경계 데이터 로드
url = 'https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/us-states.json'
state_geo = requests.get(url).json()

# 주요 State의 위도/경도 좌표
state_coords = {
    'California': [36.7783, -119.4179], 'Texas': [31.9686, -99.9018],
    'Florida': [27.6648, -81.5158], 'New York': [40.7128, -74.0060],
    'Pennsylvania': [41.2033, -77.1945], 'Illinois': [40.6331, -89.3985],
    'Ohio': [40.4173, -82.9071], 'Georgia': [32.1656, -82.9001],
    'North Carolina': [35.7596, -79.0193], 'Michigan': [44.3148, -85.6024],
    'Washington': [47.7511, -120.7401], 'Arizona': [34.0489, -111.0937],
    'Massachusetts': [42.4072, -71.3824], 'Tennessee': [35.5175, -86.5804],
    'Indiana': [40.2672, -86.1349], 'Missouri': [37.9643, -91.8318],
    'Maryland': [39.0458, -76.6413], 'Wisconsin': [43.7844, -88.7879],
    'Colorado': [39.5501, -105.7821]
}

# 지도 객체 생성
m = folium.Map(location=[37.0902, -95.7129], zoom_start=4)

# 단계구분도 그리기
folium.Choropleth(
    geo_data=state_geo,
    data=state_churn_analysis,
    columns=['state', 'churn_rate_pct'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    nan_fill_color='lightgrey',
    legend_name='Customer Churn Rate (%)'
).add_to(m)

# 이탈률 TOP 3 지역에 마커 추가
top_3_states = state_churn_analysis.head(3)

for index, row in top_3_states.iterrows():
    state_name = row['state']
    churn_val = row['churn_rate_pct']

    # 좌표 데이터가 있는 경우에만 지도 위에 마커핀를 찍음
    if state_name in state_coords:
        folium.Marker(
            location=state_coords[state_name],
            popup=f"위험 지역: {state_name} ({churn_val}%)",
            icon=folium.Icon(color='red', icon='info-sign')
        ).add_to(m)

    # 콘솔에도 정보 출력
    print(f"위험 지역 강조: {state_name} ({churn_val}%)")

# 6. 최종 지도 출력
display(m)